In [1]:
# sort.ipynb (copy these cells into a new notebook named sort.ipynb)

# --- Cell 1: Setup ---
from pathlib import Path
import pandas as pd
import tempfile
import os

CSV_PATH = Path("lab_cleaned_numeric.csv")  # in your project root

# If you ALSO want to convert explicit zeros to N/A for frontend display, set this to True.
CONVERT_ZERO_TO_NA = False


# --- Cell 2: Read CSV (preserve blanks exactly) ---
# keep_default_na=False ensures blank cells stay "" (empty string) instead of becoming NaN
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

print("Rows:", len(df))
print("Columns:", len(df.columns))


# --- Cell 3: Detect price columns + convert blanks to N/A ---
# Adjust keywords if your price columns use different naming
PRICE_COL_KEYWORDS = ("price", "amount", "cost")

price_cols = [
    c for c in df.columns
    if any(k in c.lower() for k in PRICE_COL_KEYWORDS)
]

if not price_cols:
    raise ValueError(
        "No price columns detected. "
        "Rename/confirm your price column headers or update PRICE_COL_KEYWORDS."
    )

print("Detected price columns:", price_cols)

for col in price_cols:
    # Convert ONLY blank/whitespace-only cells to "N/A"
    blank_mask = df[col].astype(str).str.strip().eq("")
    df.loc[blank_mask, col] = "N/A"

    if CONVERT_ZERO_TO_NA:
        zero_mask = df[col].astype(str).str.strip().isin({"0", "0.0", "0.00", "₦0", "₦0.0", "₦0.00"})
        df.loc[zero_mask, col] = "N/A"

print("Done converting blanks to N/A in price columns.")


# --- Cell 4: Write back safely (overwrite the same file) ---
# Write to a temp file first, then replace the original (safer than direct overwrite)
with tempfile.NamedTemporaryFile("w", delete=False, suffix=".csv", newline="", encoding="utf-8") as tmp:
    tmp_path = Path(tmp.name)
    df.to_csv(tmp_path, index=False, lineterminator="\n")

os.replace(tmp_path, CSV_PATH)
print(f"Saved: {CSV_PATH.resolve()}")


Rows: 758
Columns: 8
Detected price columns: ['price_outsourced', 'price_walkin', 'price_hospital']
Done converting blanks to N/A in price columns.
Saved: C:\Users\ACEMX\Desktop\LLMprojects\e-CLINIC\lab_cleaned_numeric.csv
